# ============================================================
# STEP 2D-INDUCTIVE — PER-FOLD GRAPH FEATURE RECOMPUTATION
# Turkish Quishing Detection — TUMC
# ============================================================
# Recomputes the 18 graph features WITHOUT cross-fold leakage.
#
# TRANSDUCTIVE (Step 2D, already done): graph statistics computed
#   once over the entire corpus. Test-fold URLs influence the
#   token co-occurrence tables → optimistic upper bound.
#
# INDUCTIVE (this file): for each CV fold, the co-occurrence
#   tables are built using TRAINING-FOLD URLs ONLY. Test-fold
#   URLs are then mapped through the train-only graph, exactly
#   as a deployed system maps a never-before-seen URL through a
#   graph built from historical data. No test information leaks.
#
# Output: features_full_final_inductive.csv
#   (same 18 graph columns, but leakage-free; lexical/adv/turkish
#    columns are identical to _final and copied through)
#
# This is the honest/deployment feature set. Run Step 3 on BOTH
# final (transductive) and final_inductive to report the gap.
# ============================================================

In [ ]:
import os, re, warnings
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
SEED = 42

DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
INPUT_FILE  = os.path.join(DATA_DIR, "features_full_final.csv")   # before graph
OUTPUT_CSV  = os.path.join(DATA_DIR, "features_full_final_inductive.csv")

CAMPAIGN_MIN  = 3
RARE_THRESHOLD = 50

GRAPH_FEATURES = [
    "rare_token_count","max_token_cluster_size","shared_token_degree",
    "campaign_token_score","unique_token_ratio","token_reuse_score",
    "domain_family_size","tld_token_cooccur","sibling_domain_count",
    "domain_ngram_cluster","registrant_pattern_score",
    "subdomain_reuse_count","path_template_reuse","host_pattern_degree",
    "campaign_membership","token_centrality","is_hub_domain",
    "cluster_malicious_ratio",
]

def tok_struct(url):
    u = re.sub(r"^https?://", "", str(url).lower())
    return [p for p in re.split(r"[/.\-_?=&]", u) if len(p) >= 4 and not p.isdigit()]

def tok_domain(domain):
    return [t for t in re.split(r"[.\-_]", str(domain).lower()) if len(t) >= 3]

def build_tables(idx, urls, domains, tlds, labels):
    """Build co-occurrence tables from a given index subset (train only)."""
    tot = Counter(); dom_ct = Counter(); mal = defaultdict(int); ben = defaultdict(int)
    tld_tok = Counter(); fam = Counter(); seen = set()
    dtoks_map = {}
    for i in idx:
        toks = set(tok_struct(urls[i]))
        dtoks_map[i] = toks
        for t in toks:
            tot[t] += 1
            if labels[i] == 1: mal[t] += 1
            else: ben[t] += 1
            tld_tok[(tlds[i], t)] += 1
        d = domains[i]
        if d and d not in seen:
            seen.add(d)
            for t in tok_domain(d):
                dom_ct[t] += 1
        if len(d) >= 6:
            fam[d[:3] + d[-3:]] += 1
    return tot, dom_ct, mal, ben, tld_tok, fam

def compute_row(url, domain, tld, tot, dom_ct, mal, ben, tld_tok, fam):
    toks = set(tok_struct(url))
    f = {}
    rare = [t for t in toks if tot.get(t,0) >= CAMPAIGN_MIN
            and dom_ct.get(t, 999) < RARE_THRESHOLD]
    f["rare_token_count"] = len(rare)
    f["max_token_cluster_size"] = max((tot.get(t,0) for t in toks), default=0)
    f["shared_token_degree"] = sum(tot.get(t,0) for t in toks)
    camp = sum(1 for t in rare
               if (mal.get(t,0)+ben.get(t,0))>0
               and mal.get(t,0)/(mal.get(t,0)+ben.get(t,0))>0.8)
    f["campaign_token_score"] = camp
    nt = max(len(toks),1)
    f["unique_token_ratio"] = sum(1 for t in toks if tot.get(t,0)<=2)/nt
    f["token_reuse_score"]  = sum(1 for t in toks if tot.get(t,0)>10)/nt
    sig = domain[:3]+domain[-3:] if len(domain)>=6 else domain
    f["domain_family_size"] = fam.get(sig,1)
    f["tld_token_cooccur"]  = sum(tld_tok.get((tld,t),0) for t in toks)
    f["sibling_domain_count"] = max(fam.get(sig,1)-1,0)
    f["domain_ngram_cluster"] = sum(dom_ct.get(t,0) for t in tok_domain(domain))
    f["registrant_pattern_score"] = len(domain)+domain.count("-")*3+sum(c.isdigit() for c in domain)*2
    f["subdomain_reuse_count"] = sum(1 for t in toks if dom_ct.get(t,0)>5)
    f["path_template_reuse"]   = sum(tot.get(t,0) for t in rare)
    f["host_pattern_degree"]   = len([t for t in toks if tot.get(t,0)>=CAMPAIGN_MIN])
    f["campaign_membership"]   = int(camp>0 or f["domain_family_size"]>5)
    f["token_centrality"] = round(f["shared_token_degree"]/nt,2)
    f["is_hub_domain"]    = int(f["max_token_cluster_size"]>100)
    ms = sum(mal.get(t,0) for t in toks); ts = sum(mal.get(t,0)+ben.get(t,0) for t in toks)
    f["cluster_malicious_ratio"] = round(ms/max(ts,1),4)
    return {k: f[k] for k in GRAPH_FEATURES}

# ════════════════════════════════════════════════════════════
print("="*70)
print("STEP 2D-INDUCTIVE — PER-FOLD GRAPH FEATURES (leakage-free)")
print("="*70)

df = pd.read_csv(INPUT_FILE, low_memory=False)
n = len(df)
print(f"\n[1] Loaded {n:,} rows (pre-graph, v11)")

urls    = df["url"].astype(str).values
domains = df["reg_domain"].astype(str).values
tlds    = df["tld"].astype(str).values
labels  = df["label_enc"].values
folds   = df["fold"].values
unique_folds = sorted(df["fold"].unique())

# Allocate output graph matrix
graph_out = np.zeros((n, len(GRAPH_FEATURES)), dtype=float)

print(f"\n[2] Recomputing graph features per fold (train-only tables) ...")
for tf in unique_folds:
    train_idx = np.where(folds != tf)[0]
    test_idx  = np.where(folds == tf)[0]
    print(f"    Fold {tf}: build tables on {len(train_idx):,} train, "
          f"map {len(test_idx):,} test")
    # Build co-occurrence tables from TRAIN ONLY
    tot, dom_ct, mal, ben, tld_tok, fam = build_tables(
        train_idx, urls, domains, tlds, labels)
    # Map TEST URLs through train-only tables (no leakage)
    for i in test_idx:
        row = compute_row(urls[i], domains[i], tlds[i],
                          tot, dom_ct, mal, ben, tld_tok, fam)
        graph_out[i] = [row[k] for k in GRAPH_FEATURES]

g_df = pd.DataFrame(graph_out, columns=GRAPH_FEATURES)
print(f"\n[3] Inductive graph matrix: {g_df.shape}")

# Merge with v11 (lexical+adv+turkish) — these are unchanged
META = {"url","source","tld","label","label_enc",
        "class_final","class_enc","fold","reg_domain"}
EXISTING = [c for c in df.columns if c not in META]
full = pd.concat([df[EXISTING].reset_index(drop=True),
                  g_df.reset_index(drop=True)], axis=1)
for c in META:
    if c in df.columns:
        full[c] = df[c].values
full.to_csv(OUTPUT_CSV, index=False)
print(f"\n[4] Saved → {OUTPUT_CSV}")

# Compare transductive vs inductive MI for the headline feature
print(f"\n[5] Quick transductive-vs-inductive comparison:")
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(
    g_df[["max_token_cluster_size","domain_family_size","token_centrality"]].values,
    df["class_enc"].values, random_state=SEED)
print(f"    INDUCTIVE MI (5-class):")
for name, v in zip(["max_token_cluster_size","domain_family_size","token_centrality"], mi):
    print(f"      {name:<24s}: {v:.4f}")
print(f"    (Compare to transductive: 0.496, 0.289, 0.478)")
print(f"    Large drop = features were leaking. Small drop = genuine.")

print(f"""
{'='*70}
STEP 2D-INDUCTIVE COMPLETE
{'='*70}
  Transductive features: features_full_v12.csv          (Step 2D)
  Inductive  features:   features_full_v12_inductive.csv (this)

  Run Step 3 on BOTH and report the gap as:
    transductive = upper bound (full graph observed)
    inductive    = deployment estimate (train-only graph)
{'='*70}
""")

STEP 2D-INDUCTIVE — PER-FOLD GRAPH FEATURES (leakage-free)

[1] Loaded 1,239,308 rows (pre-graph, v11)

[2] Recomputing graph features per fold (train-only tables) ...
    Fold 0: build tables on 991,213 train, map 248,095 test
    Fold 1: build tables on 990,446 train, map 248,862 test
    Fold 2: build tables on 994,741 train, map 244,567 test
    Fold 3: build tables on 993,103 train, map 246,205 test
    Fold 4: build tables on 987,729 train, map 251,579 test

[3] Inductive graph matrix: (1239308, 18)

[4] Saved → /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/features_full_final_inductive.csv

[5] Quick transductive-vs-inductive comparison:
    INDUCTIVE MI (5-class):
      max_token_cluster_size  : 0.3865
      domain_family_size      : 0.1551
      token_centrality        : 0.3813
    (Compare to transductive: 0.496, 0.289, 0.478)
    Large drop = features were leaking. Small drop = genuine.

STEP 2D-INDUCTIVE COMPLETE
  Transductive features: features_full_v12.